# 2차시 ― 머신러닝·딥러닝 모델, 직접 써 보기

> 1차시에서 Google Colab에 접속해 코드 셀을 실행해 보았습니다. 이번 노트북은 **파이썬 문법을 몰라도** 괜찮습니다(문법은 3차시). 지금은 **잘 만들어진 AI 모델을 한두 줄로 불러와 결과를 직접 확인**하는 것이 목표예요. 코드를 한 줄씩 이해하지 못해도 됩니다. **셀을 실행하고(Shift + Enter), 나온 결과를 감상**하세요.

**오늘의 순서**
1. ⚡ **먼저 CPU vs GPU 차이를 직접 체감** — 왜 런타임을 GPU로 바꿔야 하는지 몸으로 느낍니다.
2. 🎯 **필수 6가지** — 한국어 중심으로 AI 모델을 가져다 씁니다(감성·번역·빈칸채우기·제로샷·개체명인식·이미지).
3. 🎁 **선택(시간이 남으면)** — 요약·글 이어쓰기·객체 탐지.

> 📌 ✏️ 콜아웃으로 표시된 "연습문제"는 직접 입력하는 칸입니다. 📎 콜아웃은 그 기술이 실제 어디에 쓰이는지 더 볼 수 있는 자료입니다.

## 1. ⚡ CPU vs GPU — 직접 체감하기

딥러닝 모델은 결국 **거대한 행렬 곱셈**입니다. **CPU**는 이런 계산을 순서대로 처리하고, **GPU**는 수천 개의 계산을 **동시에** 처리해 훨씬 빠릅니다. 아래 셀을 실행해 **같은 계산이 장치에 따라 얼마나 차이 나는지** 직접 측정해 봅니다.

> `torch`는 Colab에 이미 설치되어 있어 바로 실행됩니다(설치 불필요).

In [ ]:
import torch, time

device = "cuda" if torch.cuda.is_available() else "cpu"
print("현재 사용 중인 장치:", device.upper())

# 큰 행렬(3000x3000)을 여러 번 곱해 시간을 재봅니다
N, ITERS = 3000, 20
x = torch.randn(N, N, device=device)
_ = torch.mm(x, x)                      # 워밍업(첫 호출 준비)
if device == "cuda": torch.cuda.synchronize()

start = time.time()
for _ in range(ITERS):
    y = torch.mm(x, x)
if device == "cuda": torch.cuda.synchronize()
print(f"큰 행렬 곱셈 {ITERS}번 걸린 시간: {time.time() - start:.2f}초")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">⚡ 먼저 이것부터 — CPU로 한 번, GPU로 한 번</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">위 결과의 <b>장치</b>와 <b>걸린 시간</b>을 메모해 두세요. 지금은 <b>CPU</b>일 겁니다(수 초~수십 초).<br><br><b>▶ 이제 GPU로 바꿉니다:</b> 상단 메뉴 <b>런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU → 저장</b>.<br>⚠️ 런타임을 바꾸면 <b>세션이 다시 시작</b>되어 지금까지 실행한 내용이 초기화됩니다(아직 무거운 모델을 내려받기 전이라 괜찮습니다).<br>바꿨으면 바로 위 벤치마크 셀을 <b>다시 실행</b>해 시간을 비교하세요. 장치가 <code>CUDA</code>로 바뀌고 시간이 <b>확 줄어드는</b> 걸 확인할 수 있습니다.</div></div>

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">💡 GPU는 왜 필요한가</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">왜 이렇게까지 GPU를 쓸까요? 오늘처럼 작은 모델을 한 번 쓰는 건 CPU로도 가능하지만, <b>수억·수천억 개의 숫자로 된 큰 모델을 학습시키거나 몇 번이고 돌리면</b> CPU로는 몇 시간~며칠이 걸립니다. 그래서 AI 학습에는 GPU가 거의 필수예요. <b>이번 실습은 GPU를 켜 둔 상태로 진행</b>합니다 — 뒤에 나오는 제로샷 모델(2.2GB)에서 그 차이를 한 번 더 느끼게 됩니다.</div></div>

## 2. 준비 — 라이브러리 설치

Hugging Face의 `transformers`는 전 세계 연구자들이 학습해 공개한 AI 모델을 손쉽게 불러오는 도구이고, `torch`는 그 모델이 계산을 수행하는 엔진입니다. 아래 셀을 한 번만 실행해 두면 됩니다.

> `sentencepiece`·`protobuf`는 번역·제로샷 모델이, `timm`은 맨 뒤 객체 탐지 모델이 필요로 하는 보조 도구입니다. 한 번에 같이 설치합니다.

In [ ]:
!pip install -q transformers torch sentencepiece protobuf timm

모델을 불러오는 도구를 가져옵니다. `pipeline` 하나면 "어떤 일(task)을 할지"와 "어떤 모델을 쓸지"만 알려 주면 바로 쓸 수 있습니다.

아래서는 앞서 확인한 장치를 `DEVICE`에 담아두고, 모든 모델에 같이 넘겨 GPU를 쓰게 합니다.

In [ ]:
from transformers import pipeline
import torch

# GPU가 있으면 0(GPU), 없으면 -1(CPU)
DEVICE = 0 if torch.cuda.is_available() else -1
print("모델 실행 장치:", "GPU" if DEVICE == 0 else "CPU (런타임을 GPU로 바꿔 보세요)")

---

# 🎯 필수 — 한국어로 AI 가져다 쓰기 (6가지)

## 3. 감성 분석 — 이 리뷰는 몇 점짜리일까

문장을 넣으면 **별 1개(매우 부정)~별 5개(매우 긍정)**로 감정을 예측합니다. 이 모델은 한국어를 포함해 여러 언어를 이해해서, 한국어 문장을 그대로 넣을 수 있습니다.

In [ ]:
sentiment = pipeline(
    task="sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=DEVICE,
)

print(sentiment("이 영화 정말 최고였어요. 강추합니다!"))

In [ ]:
reviews = [
    "배송도 빠르고 음식도 따뜻해서 좋았어요.",
    "가격에 비해 품질이 너무 아쉬워요. 다시는 안 살 거예요.",
    "This product is absolutely wonderful!",   # 영어도 됩니다
]
for r in reviews:
    print(r, "->", sentiment(r))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>감성 분석(텍스트 분류)</b> 은(는) 이런 곳에 쓰입니다 — 쇼핑몰·앱 리뷰 별점 자동 집계, 브랜드 평판 모니터링, 고객 문의 긍·부정 분류, 스팸 필터<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/text-classification">huggingface.co/tasks/text-classification</a></div></div>

In [ ]:
# 연습문제 1: my_text 문장을 바꿔 몇 점짜리인지 확인하세요.
my_text = "서비스는 친절했지만 음식이 조금 짰어요."
print(sentiment(my_text))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 1 내 문장으로 감성 분석</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>문제</b> &nbsp; <code>my_text</code> 에 칭찬·불만 문장을 직접 써서 별점을 확인하세요.<br><b>힌트</b> &nbsp; 칭찬과 불만을 섞으면 별점이 어떻게 변하는지 보세요.<br><b>예상</b> &nbsp; <code>[{'label': '4 stars', 'score': ...}]</code> 처럼 별점이 나옵니다.</div></div>

## 4. 번역 — 한국어를 영어로

한국어 문장을 넣으면 영어로 바꿔 줍니다.

In [ ]:
translator = pipeline(
    task="translation",
    model="Helsinki-NLP/opus-mt-ko-en",
    device=DEVICE,
)

print(translator("오늘 날씨가 정말 좋아서 기분이 좋습니다.")[0]["translation_text"])

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>기계 번역</b> 은(는) 이런 곳에 쓰입니다 — 파파고·구글 번역 같은 자동 번역, 다국어 고객 상담, 해외 논문·문서 읽기<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/translation">huggingface.co/tasks/translation</a></div></div>

In [ ]:
# 연습문제 2: my_sentence 를 바꿔 영어 번역을 확인하세요.
my_sentence = "저는 인공지능을 배우는 것이 무척 즐겁습니다."
print(translator(my_sentence)[0]["translation_text"])

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 2 내 문장 번역해 보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>문제</b> &nbsp; <code>my_sentence</code> 의 한국어 문장을 바꿔 영어 번역을 확인하세요.<br><b>힌트</b> &nbsp; 짧고 간단한 문장이 더 잘 번역됩니다.<br><b>예상</b> &nbsp; 영어 번역문이 한 줄 출력됩니다.</div></div>

## 5. 빈칸 채우기 — [MASK] 자리에 들어갈 말

문장의 `[MASK]` 자리에 어울리는 단어를 모델이 확률과 함께 제안합니다. 한국어로 학습된 `klue/bert-base` 모델입니다.

In [ ]:
unmasker = pipeline(
    task="fill-mask",
    model="klue/bert-base",
    device=DEVICE,
)

for r in unmasker("저는 오늘 아침에 [MASK]을 마셨습니다.")[:3]:
    print(r["token_str"], "->", round(r["score"], 3))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>빈칸 채우기(마스크드 언어모델)</b> 은(는) 이런 곳에 쓰입니다 — 검색어·메시지 자동완성, 오타 교정, 그리고 BERT 같은 모델이 언어를 배우는 기본 원리<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/fill-mask">huggingface.co/tasks/fill-mask</a></div></div>

In [ ]:
# 연습문제 3: 문장을 바꿔 보세요. 빈칸 자리에는 꼭 [MASK] 를 남겨 두세요.
my_sentence = "주말에는 친구와 함께 [MASK]를 봤습니다."
for r in unmasker(my_sentence)[:3]:
    print(r["token_str"], "->", round(r["score"], 3))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 3 바뀐 문장으로 빈칸 채우기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>문제</b> &nbsp; 문장을 바꿔 보세요. 단, 빈칸 자리에는 꼭 <code>[MASK]</code> 를 남겨 두세요.<br><b>힌트</b> &nbsp; 앞뒤 문맥을 바꾸면 제안되는 단어도 달라집니다.<br><b>예상</b> &nbsp; 어울리는 단어 3개가 확률과 함께 나옵니다.</div></div>

## 6. 제로샷 분류 — 분류 항목을 내가 정한다

미리 학습하지 않은 항목이라도, **내가 그 자리에서 정한 분류 항목**으로 문장을 나눌 수 있습니다. 이 모델은 **2.2GB로 커서** 처음 불러올 때 시간이 좀 걸립니다.

> ⚡ 앞서 CPU였다면 이 모델은 **매우 느렸을** 겁니다. 지금 GPU라서 이 정도입니다 — 1번에서 본 차이가 실제 모델에서 드러나는 지점입니다.

In [ ]:
zero_shot = pipeline(
    task="zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=DEVICE,
)

sentence = "이번 주말에 가족과 함께 제주도로 여행을 떠납니다."
labels = ["여행", "음식", "스포츠", "정치"]

result = zero_shot(sentence, candidate_labels=labels)
for label, score in zip(result["labels"], result["scores"]):
    print(label, "->", round(score, 3))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>제로샷 분류</b> 은(는) 이런 곳에 쓰입니다 — 학습 데이터 없이 문의 사유 자동 분류, 콘텐츠 자동 태그, 새 카테고리가 생겨도 바로 대응<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/zero-shot-classification">huggingface.co/tasks/zero-shot-classification</a></div></div>

In [ ]:
# 연습문제 4: 문장과 분류 항목(라벨)을 모두 바꿔 보세요.
my_sentence = "어제 산 노트북이 너무 느려서 환불을 받고 싶습니다."
my_labels = ["칭찬", "불만", "질문"]

result = zero_shot(my_sentence, candidate_labels=my_labels)
for label, score in zip(result["labels"], result["scores"]):
    print(label, "->", round(score, 3))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 4 분류 항목을 직접 정하기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>문제</b> &nbsp; <code>my_sentence</code> 와 <code>my_labels</code> 를 모두 자유롭게 바꿔 보세요.<br><b>힌트</b> &nbsp; 라벨을 "칭찬/불만/질문" 처럼 업무에 맞게 정해 보세요.<br><b>예상</b> &nbsp; 제가 준 라벨이 점수 높은 순서로 정렬됩니다.</div></div>

## 7. 개체명 인식 (NER) — 문장 속 이름 찾기

문장에서 **사람·장소·기관·날짜** 같은 고유한 이름을 찾아 종류를 붙입니다. 자연어처리(NLP)의 대표적인 **전통 과제** 중 하나로, 한국어 모델을 씁니다.

In [ ]:
ner = pipeline(
    task="token-classification",
    model="Leo97/KoELECTRA-small-v3-modu-ner",
    aggregation_strategy="simple",
    device=DEVICE,
)

for e in ner("김찬 교수는 한경국립대학교에서 인공지능을 가르칩니다."):
    print(e["word"], "->", e["entity_group"], round(float(e["score"]), 2))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🗂️ 유형 코드 읽는 법</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">결과의 유형 코드는 이런 뜻입니다 — <b>PS</b> 사람 · <b>LC</b> 지역·장소 · <b>OG</b> 기관·단체 · <b>DT</b> 날짜 · <b>TI</b> 시간 · <b>QT</b> 수량 · <b>CV</b> 직업·문명. 이 모델은 이런 15가지 유형으로 이름을 분류합니다.</div></div>

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>개체명 인식(NER)</b> 은(는) 이런 곳에 쓰입니다 — 개인정보 비식별화(이름·주소 가리기), 뉴스·이력서에서 인물·기관 추출, 검색 엔진·챗봇의 의도 파악<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/token-classification">huggingface.co/tasks/token-classification</a></div></div>

In [ ]:
# 연습문제 5: 문장을 바꿔 이름을 찾아 보세요.
for e in ner("이순신 장군은 1592년 한산도에서 승리했습니다."):
    print(e["word"], "->", e["entity_group"], round(float(e["score"]), 2))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 5 내 문장에서 이름 찾기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>문제</b> &nbsp; 사람·장소·날짜가 들어간 문장을 직접 써서 무엇이 어떤 유형으로 잡히는지 보세요.<br><b>힌트</b> &nbsp; 유명한 인물·지명을 넣으면 잘 찾습니다.<br><b>예상</b> &nbsp; 단어별로 PS·LC·DT 같은 유형이 붙습니다.</div></div>

## 8. 이미지 분류 — AI는 글만? 사진도 봅니다

지금까지는 글을 다뤄는 모델이었습니다. AI는 **사진**도 봅니다. 사진을 넣으면 무엇인지 맞혀 봅니다.

In [ ]:
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"   # 고양이 두 마리 사진
image = Image.open(requests.get(url, stream=True).raw)
image   # 셀 마지막 줄에 변수 이름만 두면 Colab이 그림을 보여 줍니다

In [ ]:
image_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224",
    device=DEVICE,
)

for p in image_classifier(image, top_k=3):
    print(p["label"], "->", round(p["score"], 3))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>이미지 분류</b> 은(는) 이런 곳에 쓰입니다 — 사진 자동 앨범 분류, 공장 불량품 검출, 의료 영상 판독 보조<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/image-classification">huggingface.co/tasks/image-classification</a></div></div>

In [ ]:
# 연습문제 6: my_url 에 다른 사진 주소(.jpg/.png 로 끝나는 URL)를 넣어 분류해 보세요.
my_url = "http://images.cocodataset.org/val2017/000000000632.jpg"
my_image = Image.open(requests.get(my_url, stream=True).raw)
for p in image_classifier(my_image, top_k=3):
    print(p["label"], "->", round(p["score"], 3))

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습문제 6 다른 사진으로 분류하기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>문제</b> &nbsp; <code>my_url</code> 에 다른 사진 주소를 넣어 분류해 보세요(주소가 <code>.jpg</code>/<code>.png</code>로 끝나야 해요).<br><b>힌트</b> &nbsp; 구글 이미지에서 사진을 우클릭 → "이미지 주소 복사"로 주소를 얻을 수 있어요.<br><b>예상</b> &nbsp; 사진 속 사물 후보 3개가 확률과 함께 나옵니다.</div></div>

---

# 🎁 선택 — 시간이 남으면 더 해 보기

여기부터는 **여유 시간용**입니다. 수업에서 건너뛰어도 되고, 궁금하면 하나씩 실행해 보세요. 새 모델을 불러오므로 몇 초가 걸립니다.

## 9. (선택) 텍스트 요약 — 긴 글을 짧게

여러 문장으로 된 긴 글을 핵심만 남겨 줍니다. 한국어 요약 모델입니다.

In [ ]:
summarizer = pipeline(
    task="summarization",
    model="eenzeenee/t5-base-korean-summarization",
    device=DEVICE,
)

article = (
    "인공지능은 의료, 교통, 금융 등 다양한 분야에서 활용되고 있습니다. "
    "의료 분야에서는 의료 영상을 분석해 질병을 조기에 발견하도록 돕고, "
    "교통 분야에서는 자율주행 기술에 쓰입니다. "
    "많은 기업이 고객 상담에 AI 챗봇을 도입하고 있습니다."
)
print(summarizer(article, max_length=50, min_length=15)[0]["summary_text"])

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>텍스트 요약</b> 은(는) 이런 곳에 쓰입니다 — 뉴스·회의록 요약, 긴 보고서 브리핑, 긴 글 미리보기<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/summarization">huggingface.co/tasks/summarization</a></div></div>

## 10. (선택) 글 이어쓰기 — 뒷문장 만들기

앞부분을 주면 모델이 **뒤를 이어서** 써 줍니다. ChatGPT 같은 생성형 AI의 가장 기본 형태입니다. 한국어 모델(KoGPT2)입니다.

In [ ]:
generator = pipeline(
    task="text-generation",
    model="skt/kogpt2-base-v2",
    device=DEVICE,
)

print(generator("인공지능은", max_new_tokens=40)[0]["generated_text"])

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>텍스트 생성</b> 은(는) 이런 곳에 쓰입니다 — 문장 자동완성, 초안 작성 도우미, 그리고 ChatGPT 같은 대화형 AI의 뿌리<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/text-generation">huggingface.co/tasks/text-generation</a></div></div>

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">💡 왜 좀 이상하죠?</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">작은 모델이라 문장이 어색하거나 사실과 다를 수 있습니다. 이런 기본 모델을 훨씬 크게 키우고 다듬은 것이 바로 ChatGPT입니다.</div></div>

## 11. (선택) 객체 탐지 — 사진 속 물체 찾기

사진 속 물체를 찾아 **네모(박스)와 이름**으로 알려 줍니다. 시각적이라 재미있습니다. (맨 앞 설치한 `timm` 도구가 쓰입니다.)

In [ ]:
from PIL import Image
import requests

detector = pipeline(
    task="object-detection",
    model="facebook/detr-resnet-50",
    device=DEVICE,
)

url = "http://images.cocodataset.org/val2017/000000039769.jpg"   # 고양이 두 마리 사진
image = Image.open(requests.get(url, stream=True).raw)
for r in detector(image)[:6]:
    print(round(r["score"], 2), r["label"], r["box"])

<div style="border:1px solid #E7DCC6;border-left:6px solid #C79A3E;border-radius:10px;background:#FBF6EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#7A5C1E;margin-bottom:8px;">📎 어디에 쓰나 · 더 알아보기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;"><b>객체 탐지</b> 은(는) 이런 곳에 쓰입니다 — 자율주행(보행자·차량 인식), CCTV 이상 감지, 매장 재고·사람 수 세기<br>🔗 사용 사례·모델 모음: <a href="https://huggingface.co/tasks/object-detection">huggingface.co/tasks/object-detection</a></div></div>

## 12. 자주 만나는 오류와 해결

빨간 오류 메시지가 떠도 당황하지 마세요. HF 모델 실습에서 거의 다 아래에 해당합니다.

| 증상 | 원인 | 해결 |
|---|---|---|
| 첫 실행이 수십 초~몇 분 걸림 | 모델을 **처음 내려받는 중** | 정상입니다. 한 번 받으면 다음부턴 빨라요 |
| `NameError: ... pipeline ...` | 위쪽 `import` 셀을 **건너뜀** | 위(2번 설치·import)부터 순서대로 실행 |
| `ModuleNotFoundError` | 설치 셀 미실행 | 2번 설치 셀을 먼저 실행 |
| 이미지에서 오류 | URL이 사진(`.jpg`/`.png`)이 아니거나 깨짐 | 다른 이미지 주소로 교체 |
| 점점 느려지거나 메모리 부족 | 큰 모델을 여러 개 로드 | 런타임 → 세션 다시 시작 후 필요한 셀만 |
| 매우 느림(특히 제로샷·요약) | **CPU로 실행 중** | 런타임 → 런타임 유형 변경 → GPU |

> 오류 메시지의 **맨 아랫줄**이 핵심이에요. 모델 다운로드에는 인터넷이 필요합니다(Colab은 온라인).

## 13. Hugging Face 둘러보기 (코딩 없이)

오늘 쓴 모델은 모두 [huggingface.co](https://huggingface.co)에서 온 것입니다. 코딩 없이도 구경할 수 있어요.

- 🧠 **모델 허브** [huggingface.co/models](https://huggingface.co/models) — 왼쪽 `Tasks`에서 원하는 일(번역·요약·이미지 등)을 골라 모델을 둘러보세요.
- 🕹️ **Spaces** [huggingface.co/spaces](https://huggingface.co/spaces) — 다른 사람이 만든 AI 데모를 **바로 체험**해 볼 수 있어요(설치·코딩 불필요).
- 📄 **Tasks** [huggingface.co/tasks](https://huggingface.co/tasks) — 오늘 본 각 과제가 어디에 쓰이는지, 어떤 모델이 유명한지 정리되어 있어요.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;"><div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🎮 (도전) 모델을 직접 골라 쓰기</div><div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">지금까지는 제가 골라둔 모델을 썼죠. 이번엔 <b>여러분이 직접</b> 골라 봅니다.<br>1️⃣ [huggingface.co/tasks](https://huggingface.co/tasks) 에서 마음에 드는 과제를 하나 고르고,<br>2️⃣ [huggingface.co/models](https://huggingface.co/models) 에서 그 Task로 걸러 모델 이름을 하나 복사해,<br>3️⃣ 아래에 <code>pipeline(task="...", model="...", device=DEVICE)</code> 로 넣어 실행해 보세요.<br><i>(모델 크기가 작을수록 빨리 받아져요. 모델카드-모델 소개 페이지-에 예제 코드가 있기도 해요.)</i></div></div>

In [ ]:
# (도전) 예시: 허브에서 고른 한국어 글쓰기 모델을 넣어 봤습니다.
# 여러분은 다른 task·model 로 아래 두 칸을 바꿔 보세요.
my_pipe = pipeline(task="text-generation", model="skt/kogpt2-base-v2", device=DEVICE)
print(my_pipe("한경국립대학교에서 인공지능을 배우면", max_new_tokens=40)[0]["generated_text"])

## 정리 — 우리가 방금 한 일

오늘은 코드 **한두 줄**로 여러 AI 모델을 불러 직접 결과를 확인했고, **같은 계산이 CPU와 GPU에서 얼마나 차이 나는지**도 직접 측정했습니다.

| 구분 | 한 일 | task 이름 | 쓴 모델 |
|---|---|---|---|
| 필수 | 감성·별점 | `sentiment-analysis` | nlptown/bert-base-multilingual-uncased-sentiment |
| 필수 | 번역(한→영) | `translation` | Helsinki-NLP/opus-mt-ko-en |
| 필수 | 빈칸 채우기 | `fill-mask` | klue/bert-base |
| 필수 | 제로샷 분류 | `zero-shot-classification` | joeddav/xlm-roberta-large-xnli |
| 필수 | 개체명 인식 | `token-classification` | Leo97/KoELECTRA-small-v3-modu-ner |
| 필수 | 이미지 분류 | `image-classification` | google/vit-base-patch16-224 |
| 선택 | 요약 | `summarization` | eenzeenee/t5-base-korean-summarization |
| 선택 | 글 이어쓰기 | `text-generation` | skt/kogpt2-base-v2 |
| 선택 | 객체 탐지 | `object-detection` | facebook/detr-resnet-50 |

방금 `pipeline` 한 줄로 불러온 것은 **수억 개의 숫자(파라미터)로 학습된 신경망**입니다. 우리는 그 거대한 모델에게 문장이나 사진을 건네고 답을 받은 것뿐이에요.

- **3차시**에는 직접 코딩을 위한 파이썬과 수학(벡터)을 배웁니다.
- **4차시**에는 이런 모델을 (작게나마) **직접 만들어** 봅니다.
- **5차시**에는 모델을 우리 용도에 맞게 **직접 미세조정(fine-tuning)** 하고, **챗봇**을 만듭니다.

> 오늘의 핵심: AI를 꼭 밑바닥부터 만들 필요는 없습니다. 잘 만들어진 모델을 **가져다 쓰는** 것만으로도 많은 일을 할 수 있어요.